In [1]:
#Monomeric features
import pandas as pd
import numpy as np
import ast

In [2]:
df_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv')
df_train

,ID,SMILES,Permeability,Sequence,MolWt
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1216.662
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1214.646
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,"['Abu', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', '...",1202.635
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,"['Me_Bmt(E)', 'Abu', 'Sar', 'meL', 'V', 'meL',...",1202.635
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,"['A', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1188.608
...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,"['L', 'dAbu', 'A', 'L', 'dP', 'F']",626.799
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,"['L', 'dA', 'A', 'L', 'dP', 'F']",612.772
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,"['L', 'Me_dNva', 'meA', 'L', 'dP', 'meA']",606.809
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,"['L', 'Me_dAbu', 'meA', 'L', 'dP', 'meA']",592.782


In [3]:
df_train = df_train[['ID','SMILES','Sequence','Permeability']]
df_train.loc[:,'Sequence'] = df_train['Sequence'].apply(ast.literal_eval)

In [4]:
symbol_list = []

with open('/home/users/akshay/PCPpred/RRCK/data/monomer_list.txt', 'r') as file:
    for line in file:
        symbol_list.append(line.strip())
print(symbol_list)

['A', 'dA', 'meA', 'Me_dA', 'Ala(tBu)', 'Ala(indol-2-yl)', 'dAla(indol-2-yl)', 'Me_Ala(indol-2-yl)', 'Ala(5-Tet)', 'Abu', 'dAbu', 'Me_Abu', 'Me_dAbu', 'Me_Abu(morpholino)', '2Abz', 'Aib', 'Aoc(2)', '5-Ava', 'Bal', 'Me_Bal', 'HOCOCH2_Bal', 'Cys(EtO2H)_NH2', 'Cha', 'dCha', 'Me_Cha', 'D', 'meD', 'Asp_piperidide', 'Asp(OMe)', 'Asp(Ph(2-NH2))', 'dAsp(pyrrol-1-yl)', 'E', 'Glu_NH2', 'Glu(3R-Me)', 'Glu(OMe)', 'dGlu(OMe)', 'F', 'dF', 'meF', 'Me_dF', 'Phe(4-F)', 'dPhe(4-F)', 'Phe(4-CF3)', 'Phe(4-NO2)', 'Phe(CHF2)', 'dPhe(3,4-diF)', 'Et_Phe', 'H2NEt_Phe', 'Me_Phe(3-Cl)', 'Me_Phe(4-Cl)', 'Me_Phe(a,b-dehydro)', 'G', 'Bn_Gly', 'Bn(4-Cl)_Gly', 'Bn(4-OH)_Gly', 'Bu_Gly', 'iBu_Gly', 'Et_Gly', 'EtOEt_Gly', 'HOCOCH2_Gly_ol', 'MeOEt_Gly', 'NH2Bu_Gly', 'Pr_Gly', 'PhEt_Gly', 'PhPr_Gly', 'cHexCH2_Gly', 'isoamyl_Gly', 'pentyl_Gly', '3-pyridylethyl_Gly', '2-pyridylmethyl_Gly', 'd(N->O)Gly(allyl)', 'GABA', 'H', 'Hph', 'Me_Hph', 'bHph', 'Hph(2-Cl)', 'Hph(3-Cl)', 'Hph(4-Cl)', 'Hse(Et)', 'dHyp', 'Hyp(Et)', 'I', 'dI

In [5]:
def calculate_monomer_composition(df, symbol_list):
    composition_data = []

    for index, row in df.iterrows():
        # print(row['Sequence'])
        composition = {symbol: 0 for symbol in symbol_list}

        for s in row['Sequence']:
            if s in composition:
                # print(s)
                composition[s] += 1

        total_length = len(row['Sequence'])
        # print(total_length)

        frequency = {symbol: count / total_length for symbol, count in composition.items()}

        composition_data.append({
            'ID': row['ID'],
            'SMILES': row['SMILES'],
            'Permeability': row['Permeability'],
            **frequency 
        })

    composition_df = pd.DataFrame(composition_data)

    return composition_df

In [6]:
df_train_mc = calculate_monomer_composition(df_train, symbol_list)
df_train_mc.shape

(140, 388)

In [7]:
df_train_mc.to_csv("/home/users/akshay/PCPpred/RRCK/features/Monomeric/Train_mon_comp_RRCK.csv", index=False)

In [8]:
df_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv')

df_test = df_test[['ID','SMILES','Sequence','Permeability']]
df_test.loc[:,'Sequence'] = df_test['Sequence'].apply(ast.literal_eval)

df_test_mc = calculate_monomer_composition(df_test, symbol_list)
df_test_mc.shape

df_test_mc.to_csv("/home/users/akshay/PCPpred/RRCK/features/Monomeric/Test_mon_comp_RRCK.csv", index=False)
df_test_mc

,ID,SMILES,Permeability,A,dA,meA,Me_dA,Ala(tBu),Ala(indol-2-yl),dAla(indol-2-yl),...,Mono118,Mono119,Mono120,Mono121,Mono122,Mono123,Mono124,Mono125,Mono126,Mono127
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0.090909,0.090909,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0.090909,0.090909,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0.000000,0.000000,0.000000,0.200000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0.000000,0.000000,0.200000,0.100000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0.000000,0.000000,0.111111,0.222222,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0.000000,0.000000,0.125000,0.125000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0.000000,0.000000,0.000000,0.000000,0.166667,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0.000000,0.000000,0.000000,0.000000,0.166667,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
#Amino acid composition 20 natural and X as non natural amino acids

In [10]:
mono = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/CycPeptMPDB_Monomer_All.csv')
mono = mono[['Symbol','Natural_Analog']]
mono_dict = mono.set_index('Symbol')['Natural_Analog'].to_dict()
mono_dict

{'A': 'A',
 'dA': 'A',
 'meA': 'A',
 'Me_dA': 'A',
 'Ala(tBu)': 'A',
 'Ala(indol-2-yl)': 'A',
 'dAla(indol-2-yl)': 'A',
 'Me_Ala(indol-2-yl)': 'A',
 'Ala(5-Tet)': 'A',
 'Abu': 'X',
 'dAbu': 'X',
 'Me_Abu': 'X',
 'Me_dAbu': 'X',
 'Me_Abu(morpholino)': 'X',
 '2Abz': 'X',
 'Aib': 'X',
 'Aoc(2)': 'L',
 '5-Ava': 'X',
 'Bal': 'A',
 'Me_Bal': 'A',
 'HOCOCH2_Bal': 'A',
 'Cys(EtO2H)_NH2': 'C',
 'Cha': 'A',
 'dCha': 'A',
 'Me_Cha': 'A',
 'D': 'D',
 'meD': 'D',
 'Asp_piperidide': 'D',
 'Asp(OMe)': 'D',
 'Asp(Ph(2-NH2))': 'D',
 'dAsp(pyrrol-1-yl)': 'D',
 'E': 'E',
 'Glu_NH2': 'E',
 'Glu(3R-Me)': 'E',
 'Glu(OMe)': 'E',
 'dGlu(OMe)': 'E',
 'F': 'F',
 'dF': 'F',
 'meF': 'F',
 'Me_dF': 'F',
 'Phe(4-F)': 'F',
 'dPhe(4-F)': 'F',
 'Phe(4-CF3)': 'F',
 'Phe(4-NO2)': 'F',
 'Phe(CHF2)': 'F',
 'dPhe(3,4-diF)': 'F',
 'Et_Phe': 'F',
 'H2NEt_Phe': 'F',
 'Me_Phe(3-Cl)': 'F',
 'Me_Phe(4-Cl)': 'F',
 'Me_Phe(a,b-dehydro)': 'F',
 'G': 'G',
 'Bn_Gly': 'G',
 'Bn(4-Cl)_Gly': 'G',
 'Bn(4-OH)_Gly': 'G',
 'Bu_Gly': 'G',
 '

In [11]:
amino_acid_list =  list(set(mono['Natural_Analog'].to_list()))
amino_acid_list

['C',
 'Y',
 'G',
 'T',
 'A',
 'S',
 'L',
 'N',
 'P',
 'Q',
 'F',
 'I',
 'E',
 'R',
 'D',
 'V',
 'W',
 'H',
 'K',
 'M',
 'X']

In [12]:
def seq2seq(sequence):
    seq = ''
    for s in sequence:
        if s in mono_dict.keys():
            seq += mono_dict[s]
    return seq

In [13]:
def calculate_aac(df, symbol_list):
    composition_data = []

    for index, row in df.iterrows():
        # print(row['Sequence'])
        composition = {symbol: 0 for symbol in symbol_list}

        for s in row['Natural_analog_sequence']:
            if s in composition:
                # print(s)
                composition[s] += 1

        total_length = len(row['Natural_analog_sequence'])
        # print(total_length)

        frequency = {symbol: count / total_length for symbol, count in composition.items()}

        composition_data.append({
            'ID': row['ID'],
            'SMILES': row['SMILES'],
            'Permeability': row['Permeability'],
            **frequency 
        })

    composition_df = pd.DataFrame(composition_data)

    return composition_df

In [14]:
df_train = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv")
df_train.loc[:,'Sequence'] = df_train['Sequence'].apply(ast.literal_eval)
df_train.loc[:, 'Natural_analog_sequence'] = df_train['Sequence'].apply(lambda x :seq2seq(x))
df_train = calculate_aac(df_train, amino_acid_list)
print('Shape',df_train.shape)
df_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Monomeric/Train_aac_RRCK.csv',index=False)
df_train

Shape (140, 24)


,ID,SMILES,Permeability,C,Y,G,T,A,S,L,...,I,E,R,D,V,W,H,K,M,X
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0.0,0.0,0.090909,0.090909,0.181818,0.0,0.363636,...,0.0,0.0,0.0,0.0,0.272727,0.0,0.0,0.0,0.0,0.000000
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0.0,0.0,0.090909,0.000000,0.181818,0.0,0.363636,...,0.0,0.0,0.0,0.0,0.272727,0.0,0.0,0.0,0.0,0.090909
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0.0,0.0,0.090909,0.090909,0.181818,0.0,0.363636,...,0.0,0.0,0.0,0.0,0.181818,0.0,0.0,0.0,0.0,0.090909
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0.0,0.0,0.090909,0.090909,0.181818,0.0,0.363636,...,0.0,0.0,0.0,0.0,0.181818,0.0,0.0,0.0,0.0,0.090909
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0.0,0.0,0.090909,0.090909,0.272727,0.0,0.363636,...,0.0,0.0,0.0,0.0,0.181818,0.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0.0,0.0,0.000000,0.000000,0.166667,0.0,0.333333,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.166667
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0.0,0.0,0.000000,0.000000,0.333333,0.0,0.333333,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0.0,0.0,0.000000,0.000000,0.333333,0.0,0.333333,...,0.0,0.0,0.0,0.0,0.166667,0.0,0.0,0.0,0.0,0.000000
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0.0,0.0,0.000000,0.000000,0.333333,0.0,0.333333,...,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.166667


In [15]:
df_test = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv")
df_test.loc[:,'Sequence'] = df_test['Sequence'].apply(ast.literal_eval)
df_test.loc[:, 'Natural_analog_sequence'] = df_test['Sequence'].apply(lambda x :seq2seq(x))
df_test = calculate_aac(df_test, amino_acid_list)
print('Shape',df_test.shape)
df_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Monomeric/Test_aac_RRCK.csv',index=False)
df_test

Shape (35, 24)


,ID,SMILES,Permeability,C,Y,G,T,A,S,L,...,I,E,R,D,V,W,H,K,M,X
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0.000000,0.000000,0.090909,0.181818,0.181818,0.000000,0.363636,...,0.000000,0.0,0.0,0.0,0.181818,0.0,0.0,0.000000,0.0,0.000000
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0.000000,0.000000,0.090909,0.090909,0.181818,0.000000,0.363636,...,0.000000,0.0,0.0,0.0,0.181818,0.0,0.0,0.000000,0.0,0.090909
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0.000000,0.100000,0.000000,0.000000,0.200000,0.000000,0.100000,...,0.000000,0.0,0.0,0.0,0.200000,0.0,0.0,0.000000,0.0,0.300000
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0.000000,0.100000,0.000000,0.000000,0.300000,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.100000,0.0,0.0,0.000000,0.0,0.400000
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0.000000,0.111111,0.000000,0.000000,0.333333,0.000000,0.222222,...,0.000000,0.0,0.0,0.0,0.111111,0.0,0.0,0.000000,0.0,0.111111
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0.000000,0.125000,0.000000,0.000000,0.250000,0.000000,0.500000,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0.000000,0.125000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.625000,0.0,0.0,0.000000,0.0,0.125000
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0.000000,0.000000,0.000000,0.000000,0.333333,0.166667,0.000000,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.166667
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0.142857,0.000000,0.000000,0.000000,0.000000,0.000000,0.571429,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0.000000,0.000000,0.166667,0.000000,0.333333,0.000000,0.000000,...,0.000000,0.0,0.0,0.0,0.166667,0.0,0.0,0.000000,0.0,0.000000


In [16]:
#Atomic features
import pandas as pd
import numpy as np
from rdkit import Chem

In [17]:
def get_unique_atoms(smiles_list):
    unique_atoms = set()
    
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:  # Check if the molecule was created successfully
            for atom in mol.GetAtoms():
                unique_atoms.add(atom.GetSymbol())  # Add atomic symbol to the set
    
    return unique_atoms

In [18]:
def get_atomic_composition_frequency(smiles_list):
    atomic_frequencies = []
    
    for smiles in smiles_list:
        mol = Chem.MolFromSmiles(smiles)
        composition = {'Br': 0, 'C': 0, 'Cl': 0, 'F': 0, 'N': 0, 'O': 0, 'S': 0}  
        
        if mol is not None:  # Check if the molecule was created successfully
            total_atoms = mol.GetNumAtoms()  # Get total number of atoms in the molecule
            
            for atom in mol.GetAtoms():
                composition[atom.GetSymbol()] += 1  # Count occurrences of each atom
            
            # Calculate frequency
            frequency = {atom: count / total_atoms for atom, count in composition.items()}
            atomic_frequencies.append(frequency)  # Append the frequency for this SMILES
    
    return atomic_frequencies

In [19]:
#Degree of atoms
target_atoms = {'Br', 'C', 'Cl', 'F', 'N', 'O', 'S'}
def compute_target_atom_degrees(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Handle invalid SMILES

    # Initialize a dictionary to hold the highest degrees for target atoms
    degrees_dict = {atom: 0 for atom in target_atoms}

    # Get the degree of each atom and update the dictionary for target atoms
    for atom in mol.GetAtoms():
        atom_symbol = atom.GetSymbol()
        if atom_symbol in target_atoms:
            current_degree = atom.GetDegree()
            # Update the degree if the current one is higher
            if current_degree > degrees_dict[atom_symbol]:
                degrees_dict[atom_symbol] = current_degree

    return degrees_dict

In [20]:
#Bond type
def compute_bond_types_for_cyclic_peptides(df):
    
    def compute_bond_types(smiles):
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None  # Handle invalid SMILES

        bond_types = {
            'Single': 0,
            'Double': 0,
            'Triple': 0,
            'Aromatic': 0,
            'Conjugated': 0,
            'No-bond': 0
        }

        # Iterate through the bonds in the molecule
        for bond in mol.GetBonds():
            bond_order = bond.GetBondTypeAsDouble()
            if bond_order == 1.0:
                bond_types['Single'] += 1
            elif bond_order == 2.0:
                bond_types['Double'] += 1
            elif bond_order == 3.0:
                bond_types['Triple'] += 1
            elif bond_order == 1.5:
                bond_types['Aromatic'] += 1
            elif bond_order == 1.4:
                bond_types['Conjugated'] += 1
            else:
                bond_types['No-bond'] += 1

        return bond_types

    df['Bond_Types'] = df['SMILES'].apply(compute_bond_types)

    bond_types_df = df['Bond_Types'].apply(pd.Series)

    df = pd.concat([df, bond_types_df], axis=1)

    df.drop(columns=['Bond_Types'], inplace=True)

    return df

In [21]:
#Formal charges
def calculate_overall_formal_charge(smiles):
   
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Handle invalid SMILES

    overall_charge = 0
    
    # Define valence electrons for common atoms
    valence_electrons = {
        'C': 4,
        'N': 5,
        'O': 6,
        'S': 6,
        'P': 5,
        'F': 7,
        'Cl': 7,
        'Br': 7,
        'I': 7,
    }

    for atom in mol.GetAtoms():
        atom_symbol = atom.GetSymbol()
        valence = valence_electrons.get(atom_symbol, 0)
        non_bonding = atom.GetNumImplicitHs() 
        bonding = atom.GetDegree() * 2  # 2 electrons for each bond

        # Calculate formal charge
        formal_charge = valence - (non_bonding + bonding // 2)
        overall_charge += formal_charge

    return overall_charge

In [22]:
def check_aromatic_and_ring(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return (None, None)  # Handle invalid SMILES

    # Check if the molecule is aromatic
    is_aromatic = any(atom.GetIsAromatic() for atom in mol.GetAtoms())

    # Check if the molecule contains a ring
    ring_info = mol.GetRingInfo()
    is_in_ring = ring_info.NumRings() > 0

    return (int(is_aromatic), int(is_in_ring))

In [23]:
df_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv')
df_train

,ID,SMILES,Permeability,Sequence,MolWt
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1216.662
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1214.646
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,"['Abu', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', '...",1202.635
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,"['Me_Bmt(E)', 'Abu', 'Sar', 'meL', 'V', 'meL',...",1202.635
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,"['A', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1188.608
...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,"['L', 'dAbu', 'A', 'L', 'dP', 'F']",626.799
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,"['L', 'dA', 'A', 'L', 'dP', 'F']",612.772
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,"['L', 'Me_dNva', 'meA', 'L', 'dP', 'meA']",606.809
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,"['L', 'Me_dAbu', 'meA', 'L', 'dP', 'meA']",592.782


In [24]:
unique_atoms = get_unique_atoms(df_train['SMILES'])
unique_atoms

{'C', 'F', 'N', 'O', 'S'}

In [25]:
atomic_frequencies = get_atomic_composition_frequency(df_train['SMILES'])
frequency_df = pd.DataFrame(atomic_frequencies)
frequency_df.fillna(0, inplace=True)
frequency_df

,Br,C,Cl,F,N,O,S
0,0.0,0.732558,0.0,0.0,0.127907,0.139535,0.0
1,0.0,0.732558,0.0,0.0,0.127907,0.139535,0.0
2,0.0,0.729412,0.0,0.0,0.129412,0.141176,0.0
3,0.0,0.729412,0.0,0.0,0.129412,0.141176,0.0
4,0.0,0.726190,0.0,0.0,0.130952,0.142857,0.0
...,...,...,...,...,...,...,...
135,0.0,0.733333,0.0,0.0,0.133333,0.133333,0.0
136,0.0,0.727273,0.0,0.0,0.136364,0.136364,0.0
137,0.0,0.720930,0.0,0.0,0.139535,0.139535,0.0
138,0.0,0.714286,0.0,0.0,0.142857,0.142857,0.0


In [26]:
df_train_atomic_comp = pd.concat([df_train[['ID','SMILES','Permeability']], frequency_df], axis=1)
df_train_atomic_comp

,ID,SMILES,Permeability,Br,C,Cl,F,N,O,S
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0.0,0.732558,0.0,0.0,0.127907,0.139535,0.0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0.0,0.732558,0.0,0.0,0.127907,0.139535,0.0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0.0,0.729412,0.0,0.0,0.129412,0.141176,0.0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0.0,0.729412,0.0,0.0,0.129412,0.141176,0.0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0.0,0.726190,0.0,0.0,0.130952,0.142857,0.0
...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0.0,0.733333,0.0,0.0,0.133333,0.133333,0.0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0.0,0.727273,0.0,0.0,0.136364,0.136364,0.0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0.0,0.720930,0.0,0.0,0.139535,0.139535,0.0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0.0,0.714286,0.0,0.0,0.142857,0.142857,0.0


In [27]:
df_train_atomic_comp.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_atomic_comp_RRCK.csv', index=False)

In [28]:
df_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv')
atomic_frequencies = get_atomic_composition_frequency(df_test['SMILES'])
frequency_df = pd.DataFrame(atomic_frequencies)
frequency_df.fillna(0, inplace=True)
frequency_df

,Br,C,Cl,F,N,O,S
0,0.0,0.720930,0.0,0.000000,0.127907,0.151163,0.000000
1,0.0,0.729412,0.0,0.000000,0.129412,0.141176,0.000000
2,0.0,0.730769,0.0,0.000000,0.128205,0.141026,0.000000
3,0.0,0.716216,0.0,0.000000,0.135135,0.148649,0.000000
4,0.0,0.732394,0.0,0.000000,0.126761,0.140845,0.000000
5,0.0,0.750000,0.0,0.000000,0.117647,0.132353,0.000000
6,0.0,0.746269,0.0,0.000000,0.119403,0.134328,0.000000
7,0.0,0.738462,0.0,0.030769,0.107692,0.107692,0.015385
8,0.0,0.730159,0.0,0.000000,0.126984,0.126984,0.015873
9,0.0,0.777778,0.0,0.000000,0.111111,0.095238,0.015873


In [29]:
df_test_atomic_comp = pd.concat([df_test[['ID','SMILES','Permeability']], frequency_df], axis=1)
df_test_atomic_comp

,ID,SMILES,Permeability,Br,C,Cl,F,N,O,S
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0.0,0.720930,0.0,0.000000,0.127907,0.151163,0.000000
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0.0,0.729412,0.0,0.000000,0.129412,0.141176,0.000000
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0.0,0.730769,0.0,0.000000,0.128205,0.141026,0.000000
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0.0,0.716216,0.0,0.000000,0.135135,0.148649,0.000000
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0.0,0.732394,0.0,0.000000,0.126761,0.140845,0.000000
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0.0,0.750000,0.0,0.000000,0.117647,0.132353,0.000000
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0.0,0.746269,0.0,0.000000,0.119403,0.134328,0.000000
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0.0,0.738462,0.0,0.030769,0.107692,0.107692,0.015385
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0.0,0.730159,0.0,0.000000,0.126984,0.126984,0.015873
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0.0,0.777778,0.0,0.000000,0.111111,0.095238,0.015873


In [30]:
df_test_atomic_comp.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_atomic_comp_RRCK.csv', index=False)

In [31]:
#Degree of atoms
df_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_atomic_comp_RRCK.csv')
df_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_atomic_comp_RRCK.csv')

In [32]:
df_train['Atom_Degrees'] = df_train['SMILES'].apply(compute_target_atom_degrees)
degrees_df = df_train['Atom_Degrees'].apply(pd.Series)

degrees_df.columns = [f'Degree_{atom}' for atom in target_atoms]

df_train = pd.concat([df_train, degrees_df], axis=1)
df_train = df_train.drop('Atom_Degrees', axis=1)
df_train.to_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_atomic_comp_and_degree_RRCK.csv",index=False)
df_train

,ID,SMILES,Permeability,Br,C,Cl,F,N,O,S,Degree_N,Degree_F,Degree_C,Degree_Br,Degree_S,Degree_O,Degree_Cl
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0.0,0.732558,0.0,0.0,0.127907,0.139535,0.0,3,0,3,0,0,1,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0.0,0.732558,0.0,0.0,0.127907,0.139535,0.0,3,0,3,0,0,1,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0.0,0.729412,0.0,0.0,0.129412,0.141176,0.0,3,0,3,0,0,1,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0.0,0.729412,0.0,0.0,0.129412,0.141176,0.0,3,0,3,0,0,2,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0.0,0.726190,0.0,0.0,0.130952,0.142857,0.0,3,0,3,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0.0,0.733333,0.0,0.0,0.133333,0.133333,0.0,3,0,3,0,0,1,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0.0,0.727273,0.0,0.0,0.136364,0.136364,0.0,3,0,3,0,0,1,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0.0,0.720930,0.0,0.0,0.139535,0.139535,0.0,3,0,3,0,0,1,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0.0,0.714286,0.0,0.0,0.142857,0.142857,0.0,3,0,3,0,0,1,0


In [33]:
df_test['Atom_Degrees'] = df_test['SMILES'].apply(compute_target_atom_degrees)
degrees_df = df_test['Atom_Degrees'].apply(pd.Series)

degrees_df.columns = [f'Degree_{atom}' for atom in target_atoms]

df_test = pd.concat([df_test, degrees_df], axis=1)
df_test = df_test.drop('Atom_Degrees', axis=1)
df_test.to_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_atomic_comp_and_degree_RRCK.csv",index=False)
df_test

,ID,SMILES,Permeability,Br,C,Cl,F,N,O,S,Degree_N,Degree_F,Degree_C,Degree_Br,Degree_S,Degree_O,Degree_Cl
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0.0,0.720930,0.0,0.000000,0.127907,0.151163,0.000000,3,0,3,0,0,1,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0.0,0.729412,0.0,0.000000,0.129412,0.141176,0.000000,3,0,3,0,0,1,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0.0,0.730769,0.0,0.000000,0.128205,0.141026,0.000000,3,0,3,0,0,1,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0.0,0.716216,0.0,0.000000,0.135135,0.148649,0.000000,3,0,3,0,0,1,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0.0,0.732394,0.0,0.000000,0.126761,0.140845,0.000000,3,0,3,0,0,1,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0.0,0.750000,0.0,0.000000,0.117647,0.132353,0.000000,3,0,3,0,0,1,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0.0,0.746269,0.0,0.000000,0.119403,0.134328,0.000000,3,0,3,0,0,1,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0.0,0.738462,0.0,0.030769,0.107692,0.107692,0.015385,3,1,4,0,2,2,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0.0,0.730159,0.0,0.000000,0.126984,0.126984,0.015873,3,0,3,0,2,1,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0.0,0.777778,0.0,0.000000,0.111111,0.095238,0.015873,3,0,4,0,2,1,0


In [34]:
#CSV with only degree columns
df_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv')
df_train = df_train[['ID','SMILES','Permeability']]
df_train['Atom_Degrees'] = df_train['SMILES'].apply(compute_target_atom_degrees)
degrees_df = df_train['Atom_Degrees'].apply(pd.Series)

degrees_df.columns = [f'Degree_{atom}' for atom in target_atoms]

df_train = pd.concat([df_train, degrees_df], axis=1)
df_train = df_train.drop('Atom_Degrees', axis=1)
df_train.to_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_atomic_degrees_RRCK.csv",index=False)
df_train

,ID,SMILES,Permeability,Degree_N,Degree_F,Degree_C,Degree_Br,Degree_S,Degree_O,Degree_Cl
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,3,0,3,0,0,1,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,3,0,3,0,0,1,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,3,0,3,0,0,1,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,3,0,3,0,0,2,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,3,0,3,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,3,0,3,0,0,1,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,3,0,3,0,0,1,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,3,0,3,0,0,1,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,3,0,3,0,0,1,0


In [35]:
#CSV with only degree columns
df_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv')
df_test = df_test[['ID','SMILES','Permeability']]
df_test['Atom_Degrees'] = df_test['SMILES'].apply(compute_target_atom_degrees)
degrees_df = df_test['Atom_Degrees'].apply(pd.Series)

# Rename the columns to include the atom symbols
degrees_df.columns = [f'Degree_{atom}' for atom in target_atoms]

# Concatenate the original DataFrame with the new degrees DataFrame
df_test = pd.concat([df_test, degrees_df], axis=1)
df_test = df_test.drop('Atom_Degrees', axis=1)
df_test.to_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_atomic_degrees_RRCK.csv",index=False)
df_test

,ID,SMILES,Permeability,Degree_N,Degree_F,Degree_C,Degree_Br,Degree_S,Degree_O,Degree_Cl
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,3,0,3,0,0,1,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,3,0,3,0,0,1,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,3,0,3,0,0,1,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,3,0,3,0,0,1,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,3,0,3,0,0,1,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,3,0,3,0,0,1,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,3,0,3,0,0,1,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,3,1,4,0,2,2,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,3,0,3,0,2,1,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,3,0,4,0,2,1,0


In [36]:
#Bond_type
df_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv')
df_train = df_train[['ID','SMILES','Permeability']]
df_train = compute_bond_types_for_cyclic_peptides(df_train)
df_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_bonds_type_RRCK.csv', index=False)


In [37]:
df_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_atomic_comp_and_degree_RRCK.csv')
df_train = compute_bond_types_for_cyclic_peptides(df_train)
df_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_Atomic_comp_atomic_degree_bonds_type_RRCK.csv', index=False)
df_train

,ID,SMILES,Permeability,Br,C,Cl,F,N,O,S,...,Degree_Br,Degree_S,Degree_O,Degree_Cl,Single,Double,Triple,Aromatic,Conjugated,No-bond
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0.0,0.732558,0.0,0.0,0.127907,0.139535,0.0,...,0,0,1,0,74,12,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0.0,0.732558,0.0,0.0,0.127907,0.139535,0.0,...,0,0,1,0,73,13,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0.0,0.729412,0.0,0.0,0.129412,0.141176,0.0,...,0,0,1,0,73,12,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0.0,0.729412,0.0,0.0,0.129412,0.141176,0.0,...,0,0,2,0,73,12,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0.0,0.726190,0.0,0.0,0.130952,0.142857,0.0,...,0,0,1,0,72,12,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0.0,0.733333,0.0,0.0,0.133333,0.133333,0.0,...,0,0,1,0,35,6,0,6,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0.0,0.727273,0.0,0.0,0.136364,0.136364,0.0,...,0,0,1,0,34,6,0,6,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0.0,0.720930,0.0,0.0,0.139535,0.139535,0.0,...,0,0,1,0,38,6,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0.0,0.714286,0.0,0.0,0.142857,0.142857,0.0,...,0,0,1,0,37,6,0,0,0,0


In [38]:
df_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv')
df_test = df_test[['ID','SMILES','Permeability']]
df_test = compute_bond_types_for_cyclic_peptides(df_test)
df_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_bonds_type_RRCK.csv', index=False)
df_test

,ID,SMILES,Permeability,Single,Double,Triple,Aromatic,Conjugated,No-bond
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,74,12,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,73,12,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,64,10,0,6,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,60,10,0,6,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,58,9,0,6,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,56,8,0,6,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,55,8,0,6,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,41,6,0,23,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,51,8,0,6,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,39,6,0,23,0,0


In [39]:
df_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_atomic_comp_and_degree_RRCK.csv')
df_test = compute_bond_types_for_cyclic_peptides(df_test)
df_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_Atomic_comp_atomic_degree_bonds_type_RRCK.csv', index=False)
df_test

,ID,SMILES,Permeability,Br,C,Cl,F,N,O,S,...,Degree_Br,Degree_S,Degree_O,Degree_Cl,Single,Double,Triple,Aromatic,Conjugated,No-bond
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0.0,0.720930,0.0,0.000000,0.127907,0.151163,0.000000,...,0,0,1,0,74,12,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0.0,0.729412,0.0,0.000000,0.129412,0.141176,0.000000,...,0,0,1,0,73,12,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0.0,0.730769,0.0,0.000000,0.128205,0.141026,0.000000,...,0,0,1,0,64,10,0,6,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0.0,0.716216,0.0,0.000000,0.135135,0.148649,0.000000,...,0,0,1,0,60,10,0,6,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0.0,0.732394,0.0,0.000000,0.126761,0.140845,0.000000,...,0,0,1,0,58,9,0,6,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0.0,0.750000,0.0,0.000000,0.117647,0.132353,0.000000,...,0,0,1,0,56,8,0,6,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0.0,0.746269,0.0,0.000000,0.119403,0.134328,0.000000,...,0,0,1,0,55,8,0,6,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0.0,0.738462,0.0,0.030769,0.107692,0.107692,0.015385,...,0,2,2,0,41,6,0,23,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0.0,0.730159,0.0,0.000000,0.126984,0.126984,0.015873,...,0,2,1,0,51,8,0,6,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0.0,0.777778,0.0,0.000000,0.111111,0.095238,0.015873,...,0,2,1,0,39,6,0,23,0,0


In [40]:
#Overall formal charge
df_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv')
df_train = df_train[['ID','SMILES','Permeability']]
df_train['Overall_Formal_Charge'] = df_train['SMILES'].apply(calculate_overall_formal_charge)
df_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_formal_charge_RRCK.csv', index=False)
df_train

,ID,SMILES,Permeability,Overall_Formal_Charge
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,106
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,107
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,106
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,106
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,106
...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,60
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,60
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,54
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,54


In [41]:
df_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv')
df_test = df_test[['ID','SMILES','Permeability']]
df_test['Overall_Formal_Charge'] = df_test['SMILES'].apply(calculate_overall_formal_charge)
df_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_formal_charge_RRCK.csv', index=False)
df_test

,ID,SMILES,Permeability,Overall_Formal_Charge
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,111
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,106
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,100
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,100
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,91
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,82
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,82
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,97
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,81
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,81


In [42]:
#Aromatic and ring
df_train = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv')
df_train = df_train[['ID','SMILES','Permeability']]
df_train[['Is_Aromatic', 'Is_In_Ring']] = df_train['SMILES'].apply(check_aromatic_and_ring).apply(pd.Series)
df_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_is_ring_is_aromatic_RRCK.csv', index=False)
df_train

,ID,SMILES,Permeability,Is_Aromatic,Is_In_Ring
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,1
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,1
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,1
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,1
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,1
...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,1,1
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,1,1
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,1
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,1


In [43]:
df_test = pd.read_csv('/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv')
df_test = df_test[['ID','SMILES','Permeability']]
df_test[['Is_Aromatic', 'Is_In_Ring']] = df_test['SMILES'].apply(check_aromatic_and_ring).apply(pd.Series)
df_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_is_ring_is_aromatic_RRCK.csv', index=False)
df_test

,ID,SMILES,Permeability,Is_Aromatic,Is_In_Ring
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,1
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,1
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,1,1
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,1,1
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,1,1
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,1,1
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,1,1
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,1,1
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,1,1
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,1,1


In [44]:
df1 = pd.read_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_Atomic_comp_atomic_degree_bonds_type_RRCK.csv")
df2 = pd.read_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_formal_charge_RRCK.csv")
df3 = pd.read_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_is_ring_is_aromatic_RRCK.csv")

merged_df = pd.merge(df1, df2, on=['ID', 'SMILES', 'Permeability'], how='inner')

# Merge the result with df3
df_train = pd.merge(merged_df, df3, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Train_all_atomic_desc_RRCK.csv', index=False)

In [45]:
df1 = pd.read_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_Atomic_comp_atomic_degree_bonds_type_RRCK.csv")
df2 = pd.read_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_formal_charge_RRCK.csv")
df3 = pd.read_csv("/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_is_ring_is_aromatic_RRCK.csv")

merged_df = pd.merge(df1, df2, on=['ID', 'SMILES', 'Permeability'], how='inner')

# Merge the result with df3
df_test = pd.merge(merged_df, df3, on=['ID', 'SMILES', 'Permeability'], how='inner')
df_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Atomic/Test_all_atomic_desc_RRCK.csv', index=False)
df_test

,ID,SMILES,Permeability,Br,C,Cl,F,N,O,S,...,Degree_Cl,Single,Double,Triple,Aromatic,Conjugated,No-bond,Overall_Formal_Charge,Is_Aromatic,Is_In_Ring
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0.0,0.720930,0.0,0.000000,0.127907,0.151163,0.000000,...,0,74,12,0,0,0,0,111,0,1
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0.0,0.729412,0.0,0.000000,0.129412,0.141176,0.000000,...,0,73,12,0,0,0,0,106,0,1
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0.0,0.730769,0.0,0.000000,0.128205,0.141026,0.000000,...,0,64,10,0,6,0,0,100,1,1
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0.0,0.716216,0.0,0.000000,0.135135,0.148649,0.000000,...,0,60,10,0,6,0,0,100,1,1
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0.0,0.732394,0.0,0.000000,0.126761,0.140845,0.000000,...,0,58,9,0,6,0,0,91,1,1
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0.0,0.750000,0.0,0.000000,0.117647,0.132353,0.000000,...,0,56,8,0,6,0,0,82,1,1
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0.0,0.746269,0.0,0.000000,0.119403,0.134328,0.000000,...,0,55,8,0,6,0,0,82,1,1
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0.0,0.738462,0.0,0.030769,0.107692,0.107692,0.015385,...,0,41,6,0,23,0,0,97,1,1
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0.0,0.730159,0.0,0.000000,0.126984,0.126984,0.015873,...,0,51,8,0,6,0,0,81,1,1
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0.0,0.777778,0.0,0.000000,0.111111,0.095238,0.015873,...,0,39,6,0,23,0,0,81,1,1


In [46]:
#Fingerprints

In [47]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem, rdFingerprintGenerator, rdMolDescriptors
from padelpy import padeldescriptor

In [48]:
df_train = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv")
df_train

,ID,SMILES,Permeability,Sequence,MolWt
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1216.662
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,"['V', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1214.646
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,"['Abu', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', '...",1202.635
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,"['Me_Bmt(E)', 'Abu', 'Sar', 'meL', 'V', 'meL',...",1202.635
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,"['A', 'Sar', 'meL', 'V', 'meL', 'A', 'dA', 'me...",1188.608
...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,"['L', 'dAbu', 'A', 'L', 'dP', 'F']",626.799
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,"['L', 'dA', 'A', 'L', 'dP', 'F']",612.772
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,"['L', 'Me_dNva', 'meA', 'L', 'dP', 'meA']",606.809
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,"['L', 'Me_dAbu', 'meA', 'L', 'dP', 'meA']",592.782


In [49]:
df_train_cmfp = df_train[['ID','SMILES','Permeability']]
df_train_cmfp

,ID,SMILES,Permeability
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87
...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51


In [50]:
def generate_count_morgan_fp(smiles, radius=2, nBits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        # Count Morgan fingerprint generator
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=nBits)
        count_fp = mfpgen.GetCountFingerprint(mol)
        
        # Convert the count fingerprint to a dense array
        dense_fp = np.zeros((nBits,), dtype=int)
        for bit, count in count_fp.GetNonzeroElements().items():
            dense_fp[bit] = count
            
        return dense_fp
    else:
        return None

In [51]:
df_train_cmfp.loc[:,'count_morgan_fp'] = df_train_cmfp['SMILES'].apply(generate_count_morgan_fp)
df_train_cmfp['count_morgan_fp']

/tmp/ipykernel_2001246/1551993060.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_cmfp.loc[:,'count_morgan_fp'] = df_train_cmfp['SMILES'].apply(generate_count_morgan_fp)


0      [0, 9, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
1      [0, 8, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
2      [0, 8, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3      [0, 7, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
4      [0, 8, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
                             ...                        
135    [0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
136    [0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
137    [0, 2, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
138    [0, 2, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
139    [0, 2, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
Name: count_morgan_fp, Length: 140, dtype: object

In [52]:
df = pd.DataFrame(df_train_cmfp['count_morgan_fp'].tolist(), index=df_train_cmfp.index)

df_train_cmfp = pd.concat([df_train_cmfp, df.add_prefix('count_fp_')], axis=1)
df_train_cmfp.shape

(140, 2052)

In [53]:
df_train_cmfp.drop('count_morgan_fp', axis=1, inplace=True)
df_train_cmfp.shape

(140, 2051)

In [54]:
df_train_cmfp.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/count_morgan_fp_train_RRCK.csv', index=False)

In [55]:
df_test = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv")
df_test_cmfp = df_test[['ID','SMILES','Permeability']]
df_test_cmfp.loc[:,'count_morgan_fp'] = df_test_cmfp['SMILES'].apply(generate_count_morgan_fp)
df_test_cmfp['count_morgan_fp']
df = pd.DataFrame(df_test_cmfp['count_morgan_fp'].tolist(), index=df_test_cmfp.index)

df_test_cmfp = pd.concat([df_test_cmfp, df.add_prefix('count_fp_')], axis=1)
df_test_cmfp.drop('count_morgan_fp', axis=1, inplace=True)
df_test_cmfp.shape

/tmp/ipykernel_2001246/1721296512.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test_cmfp.loc[:,'count_morgan_fp'] = df_test_cmfp['SMILES'].apply(generate_count_morgan_fp)


(35, 2051)

In [56]:
df_test_cmfp.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/count_morgan_fp_test_RRCK.csv', index=False)

In [57]:
def generate_fingerprints(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        # Create a Morgan fingerprint generator
        mfpgen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        
        # Generate the Morgan fingerprint as a bit vector
        morgan_fp = mfpgen.GetFingerprint(mol)
        
        # Convert the bit vector to a dense array
        dense_fp = np.zeros((2048,), dtype=int)
        for bit in range(2048):
            dense_fp[bit] = morgan_fp[bit]
        
        return dense_fp
    else:
        print("Invalid SMILES string.")
        return None

In [58]:
df_train = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv")
df_mfp_train = df_train[['ID','SMILES','Permeability']]
df_mfp_train.loc[:,'Morganfingerprints'] = df_mfp_train['SMILES'].apply(generate_fingerprints)
df = pd.DataFrame(df_mfp_train['Morganfingerprints'].tolist(), index=df_mfp_train.index)

df_mfp_train = pd.concat([df_mfp_train, df.add_prefix('Morgan_fp_')], axis=1)
df_mfp_train.drop('Morganfingerprints', axis=1, inplace=True)
df_mfp_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/morgan_fp_train_RRCK.csv', index=False)
print(df_mfp_train.shape)
df_mfp_train

/tmp/ipykernel_2001246/1927033210.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mfp_train.loc[:,'Morganfingerprints'] = df_mfp_train['SMILES'].apply(generate_fingerprints)


(140, 2051)


,ID,SMILES,Permeability,Morgan_fp_0,Morgan_fp_1,Morgan_fp_2,Morgan_fp_3,Morgan_fp_4,Morgan_fp_5,Morgan_fp_6,...,Morgan_fp_2038,Morgan_fp_2039,Morgan_fp_2040,Morgan_fp_2041,Morgan_fp_2042,Morgan_fp_2043,Morgan_fp_2044,Morgan_fp_2045,Morgan_fp_2046,Morgan_fp_2047
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,1,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,1
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [59]:
df_test = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv")
df_mfp_test = df_test[['ID','SMILES','Permeability']]
df_mfp_test.loc[:,'Morganfingerprints'] = df_mfp_test['SMILES'].apply(generate_fingerprints)
df = pd.DataFrame(df_mfp_test['Morganfingerprints'].tolist(), index=df_mfp_test.index)

df_mfp_test = pd.concat([df_mfp_test, df.add_prefix('Morgan_fp_')], axis=1)
df_mfp_test.drop('Morganfingerprints', axis=1, inplace=True)
df_mfp_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/morgan_fp_test_RRCK.csv', index=False)
print(df_mfp_test.shape)
df_mfp_test

(35, 2051)


/tmp/ipykernel_2001246/1041046336.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_mfp_test.loc[:,'Morganfingerprints'] = df_mfp_test['SMILES'].apply(generate_fingerprints)


,ID,SMILES,Permeability,Morgan_fp_0,Morgan_fp_1,Morgan_fp_2,Morgan_fp_3,Morgan_fp_4,Morgan_fp_5,Morgan_fp_6,...,Morgan_fp_2038,Morgan_fp_2039,Morgan_fp_2040,Morgan_fp_2041,Morgan_fp_2042,Morgan_fp_2043,Morgan_fp_2044,Morgan_fp_2045,Morgan_fp_2046,Morgan_fp_2047
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0,0,0,0,0,1,0,...,0,0,0,0,1,0,0,0,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0


In [60]:
#Padel fingerprints
import glob
import os
directory = '/home/users/akshay/PCPpred/fingerprints_xml'
pattern = '*.xml'  

xml_files = glob.glob(os.path.join(directory, pattern))
xml_files.sort()
xml_files

['/home/users/akshay/PCPpred/fingerprints_xml/AtomPairs2DFingerprintCount.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/AtomPairs2DFingerprinter.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/EStateFingerprinter.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/ExtendedFingerprinter.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/Fingerprinter.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/GraphOnlyFingerprinter.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/KlekotaRothFingerprintCount.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/KlekotaRothFingerprinter.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/MACCSFingerprinter.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/PubchemFingerprinter.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/SubstructureFingerprintCount.xml',
 '/home/users/akshay/PCPpred/fingerprints_xml/SubstructureFingerprinter.xml']

In [61]:
FP_list = ['AtomPairs2DCount',
 'AtomPairs2D',
 'EState',
 'Extended',
 'Fingerprinter',
 'Graphonly',
 'KlekotaRothCount',
 'KlekotaRoth',
 'MACCS',
 'PubChem',
 'SubstructureCount',
 'Substructure']

In [62]:
fp = dict(zip(FP_list, xml_files))
fp

{'AtomPairs2DCount': '/home/users/akshay/PCPpred/fingerprints_xml/AtomPairs2DFingerprintCount.xml',
 'AtomPairs2D': '/home/users/akshay/PCPpred/fingerprints_xml/AtomPairs2DFingerprinter.xml',
 'EState': '/home/users/akshay/PCPpred/fingerprints_xml/EStateFingerprinter.xml',
 'Extended': '/home/users/akshay/PCPpred/fingerprints_xml/ExtendedFingerprinter.xml',
 'Fingerprinter': '/home/users/akshay/PCPpred/fingerprints_xml/Fingerprinter.xml',
 'Graphonly': '/home/users/akshay/PCPpred/fingerprints_xml/GraphOnlyFingerprinter.xml',
 'KlekotaRothCount': '/home/users/akshay/PCPpred/fingerprints_xml/KlekotaRothFingerprintCount.xml',
 'KlekotaRoth': '/home/users/akshay/PCPpred/fingerprints_xml/KlekotaRothFingerprinter.xml',
 'MACCS': '/home/users/akshay/PCPpred/fingerprints_xml/MACCSFingerprinter.xml',
 'PubChem': '/home/users/akshay/PCPpred/fingerprints_xml/PubchemFingerprinter.xml',
 'SubstructureCount': '/home/users/akshay/PCPpred/fingerprints_xml/SubstructureFingerprintCount.xml',
 'Substruct

In [63]:
#Substructure
fingerprint = 'Substructure'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
descriptors

,Name,SubFP1,SubFP2,SubFP3,SubFP4,SubFP5,SubFP6,SubFP7,SubFP8,SubFP9,...,SubFP298,SubFP299,SubFP300,SubFP301,SubFP302,SubFP303,SubFP304,SubFP305,SubFP306,SubFP307
0,2358,1,1,1,0,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
1,2359,1,1,1,0,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
2,5669,1,1,1,0,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
3,2360,1,1,1,0,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
4,2353,1,1,1,0,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,1,1,1,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
136,2334,1,1,1,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
137,2305,1,1,1,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
138,2304,1,1,1,0,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1


In [64]:
df_Substructure_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
df_Substructure_train.head()

,ID,SMILES,Permeability,Name,SubFP1,SubFP2,SubFP3,SubFP4,SubFP5,SubFP6,...,SubFP298,SubFP299,SubFP300,SubFP301,SubFP302,SubFP303,SubFP304,SubFP305,SubFP306,SubFP307
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,2358,1,1,1,0,1,0,...,0,0,1,1,1,0,0,0,0,1
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,2359,1,1,1,0,1,0,...,0,0,1,1,1,0,0,0,0,1
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,5669,1,1,1,0,1,0,...,0,0,1,1,1,0,0,0,0,1
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,2360,1,1,1,0,1,0,...,0,0,1,1,1,0,0,0,0,1
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,2353,1,1,1,0,1,0,...,0,0,1,1,1,0,0,0,0,1


In [65]:
df_Substructure_train.drop(['Name'],axis=1,inplace=True)
df_Substructure_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/Substructure_train_RRCK.csv', index=False)
df_Substructure_train

,ID,SMILES,Permeability,SubFP1,SubFP2,SubFP3,SubFP4,SubFP5,SubFP6,SubFP7,...,SubFP298,SubFP299,SubFP300,SubFP301,SubFP302,SubFP303,SubFP304,SubFP305,SubFP306,SubFP307
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,1,1,1,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,1,1,1,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,1,1,1,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,1,1,1,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,1,1,1,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,1,1,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,1,1,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,1,1,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,1,1,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1


In [66]:
fingerprint = 'Substructure'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_Substructure_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
df_Substructure_test.drop(['Name'],axis=1,inplace=True)
df_Substructure_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/Substructure_test_RRCK.csv', index=False)
df_Substructure_test

,ID,SMILES,Permeability,SubFP1,SubFP2,SubFP3,SubFP4,SubFP5,SubFP6,SubFP7,...,SubFP298,SubFP299,SubFP300,SubFP301,SubFP302,SubFP303,SubFP304,SubFP305,SubFP306,SubFP307
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,1,1,1,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,1,1,1,0,1,0,0,...,0,0,1,1,1,0,0,0,0,1
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,1,1,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,1,1,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,1,1,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,1,1,1,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,1,1,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,1,1,0,1,0,0,0,...,0,0,1,1,1,0,0,0,0,1
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,1,1,0,0,0,0,0,...,0,0,1,1,1,0,0,0,0,1
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,1,1,0,1,0,0,0,...,0,0,1,1,1,0,0,0,0,1


In [67]:
#CountSubstructure
fingerprint = 'SubstructureCount'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_SubstructureCount_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
df_SubstructureCount_train.head()

,ID,SMILES,Permeability,Name,SubFPC1,SubFPC2,SubFPC3,SubFPC4,SubFPC5,SubFPC6,...,SubFPC298,SubFPC299,SubFPC300,SubFPC301,SubFPC302,SubFPC303,SubFPC304,SubFPC305,SubFPC306,SubFPC307
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,2358,18.0,5.0,8.0,0.0,1.0,0.0,...,0.0,0.0,67.0,67.0,15.0,0.0,0.0,0.0,0.0,43.0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,2359,18.0,5.0,8.0,0.0,1.0,0.0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,43.0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,5669,17.0,6.0,7.0,0.0,1.0,0.0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,42.0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,2360,17.0,6.0,7.0,0.0,1.0,0.0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,42.0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,2353,17.0,5.0,7.0,0.0,1.0,0.0,...,0.0,0.0,65.0,65.0,14.0,0.0,0.0,0.0,0.0,42.0


In [68]:
df_SubstructureCount_train.drop(['Name'],axis=1,inplace=True)
df_SubstructureCount_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/SubstructureCount_train_RRCK.csv', index=False)
df_SubstructureCount_train

,ID,SMILES,Permeability,SubFPC1,SubFPC2,SubFPC3,SubFPC4,SubFPC5,SubFPC6,SubFPC7,...,SubFPC298,SubFPC299,SubFPC300,SubFPC301,SubFPC302,SubFPC303,SubFPC304,SubFPC305,SubFPC306,SubFPC307
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,18.0,5.0,8.0,0.0,1.0,0.0,0.0,...,0.0,0.0,67.0,67.0,15.0,0.0,0.0,0.0,0.0,43.0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,18.0,5.0,8.0,0.0,1.0,0.0,0.0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,43.0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,17.0,6.0,7.0,0.0,1.0,0.0,0.0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,42.0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,17.0,6.0,7.0,0.0,1.0,0.0,0.0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,42.0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,17.0,5.0,7.0,0.0,1.0,0.0,0.0,...,0.0,0.0,65.0,65.0,14.0,0.0,0.0,0.0,0.0,42.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,6.0,6.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,34.0,34.0,7.0,0.0,0.0,0.0,0.0,26.0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,6.0,5.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,33.0,33.0,6.0,0.0,0.0,0.0,0.0,26.0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,7.0,6.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,34.0,34.0,6.0,0.0,0.0,0.0,0.0,20.0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,7.0,5.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,33.0,33.0,5.0,0.0,0.0,0.0,0.0,20.0


In [69]:
fingerprint = 'SubstructureCount'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) #Substructure.csv
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_SubstructureCount_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
df_SubstructureCount_test.drop(['Name'],axis=1,inplace=True)
df_SubstructureCount_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/SubstructureCount_test_RRCK.csv', index=False)
df_SubstructureCount_test

,ID,SMILES,Permeability,SubFPC1,SubFPC2,SubFPC3,SubFPC4,SubFPC5,SubFPC6,SubFPC7,...,SubFPC298,SubFPC299,SubFPC300,SubFPC301,SubFPC302,SubFPC303,SubFPC304,SubFPC305,SubFPC306,SubFPC307
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,17.0,5.0,7.0,0.0,1.0,0.0,0.0,...,0.0,0.0,67.0,67.0,15.0,0.0,0.0,0.0,0.0,43.0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,17.0,6.0,7.0,0.0,1.0,0.0,0.0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,42.0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,8.0,13.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,54.0,54.0,12.0,0.0,0.0,0.0,0.0,36.0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,8.0,9.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,50.0,50.0,8.0,0.0,0.0,0.0,0.0,36.0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,7.0,12.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,49.0,49.0,11.0,0.0,0.0,0.0,0.0,33.0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,10.0,7.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,48.0,48.0,10.0,0.0,0.0,0.0,0.0,34.0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,6.0,14.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,47.0,47.0,13.0,0.0,0.0,0.0,0.0,30.0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,3.0,7.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,37.0,37.0,14.0,0.0,0.0,0.0,0.0,39.0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,4.0,15.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,48.0,48.0,16.0,0.0,0.0,0.0,0.0,29.0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,3.0,10.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,37.0,37.0,14.0,0.0,0.0,0.0,0.0,39.0


In [70]:
#MACCS
fingerprint = 'MACCS'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_MACCS_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
df_MACCS_train.head()
df_MACCS_train.drop(['Name'],axis=1,inplace=True)
df_MACCS_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/MACCS_train_RRCK.csv', index=False)
df_MACCS_train

,ID,SMILES,Permeability,MACCSFP1,MACCSFP2,MACCSFP3,MACCSFP4,MACCSFP5,MACCSFP6,MACCSFP7,...,MACCSFP157,MACCSFP158,MACCSFP159,MACCSFP160,MACCSFP161,MACCSFP162,MACCSFP163,MACCSFP164,MACCSFP165,MACCSFP166
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,0,0,0,0,0,0,...,1,1,1,1,1,0,0,1,1,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,0,0,0,0,0,0,...,0,1,1,1,1,0,0,1,1,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,0,0,0,0,0,0,...,1,1,1,1,1,0,0,1,1,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,0,0,0,0,0,0,...,1,1,1,1,1,0,0,1,1,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,0,0,0,0,0,0,...,1,1,1,1,1,0,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,0,0,0,0,0,0,...,0,1,1,1,1,0,0,1,1,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,0,0,0,0,0,0,...,0,1,1,1,1,0,0,1,1,0


In [71]:
fingerprint = 'MACCS'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_MACCS_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
df_MACCS_test.drop(['Name'],axis=1,inplace=True)
df_MACCS_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/MACCS_test_RRCK.csv', index=False)
df_MACCS_test

,ID,SMILES,Permeability,MACCSFP1,MACCSFP2,MACCSFP3,MACCSFP4,MACCSFP5,MACCSFP6,MACCSFP7,...,MACCSFP157,MACCSFP158,MACCSFP159,MACCSFP160,MACCSFP161,MACCSFP162,MACCSFP163,MACCSFP164,MACCSFP165,MACCSFP166
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,0,0,0,0,0,0,...,1,1,1,1,1,0,0,1,1,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,0,0,0,0,0,0,...,1,1,1,1,1,0,0,1,1,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0,0,0,0,0,0,0,...,1,1,1,1,1,1,1,1,1,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,0


In [72]:
#PubChem
fingerprint = 'PubChem'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_PubChem_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
df_PubChem_train.drop(['Name'],axis=1,inplace=True)
df_PubChem_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/PubChem_train_RRCK.csv', index=False)
df_PubChem_train

,ID,SMILES,Permeability,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [73]:
fingerprint = 'PubChem'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_PubChem_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
df_PubChem_test.drop(['Name'],axis=1,inplace=True)
df_PubChem_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/PubChem_test_RRCK.csv', index=False)
df_PubChem_test

,ID,SMILES,Permeability,PubchemFP0,PubchemFP1,PubchemFP2,PubchemFP3,PubchemFP4,PubchemFP5,PubchemFP6,...,PubchemFP871,PubchemFP872,PubchemFP873,PubchemFP874,PubchemFP875,PubchemFP876,PubchemFP877,PubchemFP878,PubchemFP879,PubchemFP880
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [74]:
#AtomPairs2DCount
fingerprint = 'AtomPairs2DCount'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_AtomPairs2DCount_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
df_AtomPairs2DCount_train.drop(['Name'],axis=1,inplace=True)
df_AtomPairs2DCount_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/AtomPairs2DCount_train_RRCK.csv', index=False)
df_AtomPairs2DCount_train

,ID,SMILES,Permeability,APC2D1_C_C,APC2D1_C_N,APC2D1_C_O,APC2D1_C_S,APC2D1_C_P,APC2D1_C_F,APC2D1_C_Cl,...,APC2D10_I_I,APC2D10_I_B,APC2D10_I_Si,APC2D10_I_X,APC2D10_B_B,APC2D10_B_Si,APC2D10_B_X,APC2D10_Si_Si,APC2D10_Si_X,APC2D10_X_X
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,45.0,29.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,45.0,29.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,44.0,29.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,44.0,28.0,13.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,43.0,29.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,28.0,13.0,6.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,27.0,13.0,6.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,22.0,16.0,6.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,21.0,16.0,6.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [75]:
fingerprint = 'AtomPairs2DCount'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_AtomPairs2DCount_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
df_AtomPairs2DCount_test.drop(['Name'],axis=1,inplace=True)
df_AtomPairs2DCount_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/AtomPairs2DCount_test_RRCK.csv', index=False)
df_AtomPairs2DCount_test

,ID,SMILES,Permeability,APC2D1_C_C,APC2D1_C_N,APC2D1_C_O,APC2D1_C_S,APC2D1_C_P,APC2D1_C_F,APC2D1_C_Cl,...,APC2D10_I_I,APC2D10_I_B,APC2D10_I_Si,APC2D10_I_X,APC2D10_B_B,APC2D10_B_Si,APC2D10_B_X,APC2D10_Si_Si,APC2D10_Si_X,APC2D10_X_X
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,44.0,29.0,13.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,44.0,29.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,39.0,30.0,11.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,35.0,30.0,11.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,36.0,27.0,10.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,37.0,24.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,36.0,24.0,9.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,42.0,16.0,8.0,2.0,0.0,2.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,36.0,19.0,8.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,44.0,16.0,6.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [76]:
#AtomPairs2D
fingerprint = 'AtomPairs2D'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_AtomPairs2D_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
df_AtomPairs2D_train.drop(['Name'],axis=1,inplace=True)
df_AtomPairs2D_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/AtomPairs2D_train_RRCK.csv', index=False)
df_AtomPairs2D_train

,ID,SMILES,Permeability,AD2D1,AD2D2,AD2D3,AD2D4,AD2D5,AD2D6,AD2D7,...,AD2D771,AD2D772,AD2D773,AD2D774,AD2D775,AD2D776,AD2D777,AD2D778,AD2D779,AD2D780
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [77]:
fingerprint = 'AtomPairs2D'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
df_AtomPairs2D_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
df_AtomPairs2D_test.drop(['Name'],axis=1,inplace=True)
df_AtomPairs2D_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/AtomPairs2D_test_RRCK.csv', index=False)
df_AtomPairs2D_test

,ID,SMILES,Permeability,AD2D1,AD2D2,AD2D3,AD2D4,AD2D5,AD2D6,AD2D7,...,AD2D771,AD2D772,AD2D773,AD2D774,AD2D775,AD2D776,AD2D777,AD2D778,AD2D779,AD2D780
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,1,1,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [78]:
#EState
fingerprint = 'EState'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
EState_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
EState_train.drop(['Name'],axis=1,inplace=True)
EState_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/EState_train_RRCK.csv', index=False)
EState_train

,ID,SMILES,Permeability,EStateFP1,EStateFP2,EStateFP3,EStateFP4,EStateFP5,EStateFP6,EStateFP7,...,EStateFP70,EStateFP71,EStateFP72,EStateFP73,EStateFP74,EStateFP75,EStateFP76,EStateFP77,EStateFP78,EStateFP79
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [79]:
fingerprint = 'EState'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
EState_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
EState_test.drop(['Name'],axis=1,inplace=True)
EState_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/EState_test_RRCK.csv', index=False)
EState_test

,ID,SMILES,Permeability,EStateFP1,EStateFP2,EStateFP3,EStateFP4,EStateFP5,EStateFP6,EStateFP7,...,EStateFP70,EStateFP71,EStateFP72,EStateFP73,EStateFP74,EStateFP75,EStateFP76,EStateFP77,EStateFP78,EStateFP79
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


In [80]:
#Extended
fingerprint = 'Extended'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
Extended_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
Extended_train.drop(['Name'],axis=1,inplace=True)
Extended_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/Extended_train_RRCK.csv', index=False)
Extended_train

,ID,SMILES,Permeability,ExtFP1,ExtFP2,ExtFP3,ExtFP4,ExtFP5,ExtFP6,ExtFP7,...,ExtFP1015,ExtFP1016,ExtFP1017,ExtFP1018,ExtFP1019,ExtFP1020,ExtFP1021,ExtFP1022,ExtFP1023,ExtFP1024
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [81]:
fingerprint = 'Extended'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
Extended_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
Extended_test.drop(['Name'],axis=1,inplace=True)
Extended_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/Extended_test_RRCK.csv', index=False)
Extended_test

,ID,SMILES,Permeability,ExtFP1,ExtFP2,ExtFP3,ExtFP4,ExtFP5,ExtFP6,ExtFP7,...,ExtFP1015,ExtFP1016,ExtFP1017,ExtFP1018,ExtFP1019,ExtFP1020,ExtFP1021,ExtFP1022,ExtFP1023,ExtFP1024
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0,1,0,1,0,0,1,...,1,0,0,0,0,0,0,0,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0,1,0,0,1,0,0,...,1,0,0,0,0,0,0,0,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0


In [82]:
#Fingerprinter
fingerprint = 'Fingerprinter'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
Fingerprinter_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
Fingerprinter_train.drop(['Name'],axis=1,inplace=True)
Fingerprinter_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/Fingerprinter_train_RRCK.csv', index=False)
Fingerprinter_train

,ID,SMILES,Permeability,FP1,FP2,FP3,FP4,FP5,FP6,FP7,...,FP1015,FP1016,FP1017,FP1018,FP1019,FP1020,FP1021,FP1022,FP1023,FP1024
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,0,0,0,0,1,0,...,1,1,0,0,0,0,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0


In [83]:
fingerprint = 'Fingerprinter'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
Fingerprinter_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
Fingerprinter_test.drop(['Name'],axis=1,inplace=True)
Fingerprinter_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/Fingerprinter_test_RRCK.csv', index=False)
Fingerprinter_test

,ID,SMILES,Permeability,FP1,FP2,FP3,FP4,FP5,FP6,FP7,...,FP1015,FP1016,FP1017,FP1018,FP1019,FP1020,FP1021,FP1022,FP1023,FP1024
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0,0,0,0,1,0,0,...,0,1,0,1,1,0,0,1,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,1,0,0,0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0,0,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0


In [84]:
#GraphOnly
fingerprint = 'Graphonly'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
Graphonly_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
Graphonly_train.drop(['Name'],axis=1,inplace=True)
Graphonly_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/Graphonly_train_RRCK.csv', index=False)
Graphonly_train

,ID,SMILES,Permeability,GraphFP1,GraphFP2,GraphFP3,GraphFP4,GraphFP5,GraphFP6,GraphFP7,...,GraphFP1015,GraphFP1016,GraphFP1017,GraphFP1018,GraphFP1019,GraphFP1020,GraphFP1021,GraphFP1022,GraphFP1023,GraphFP1024
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,0,0,0,0,0,0,...,1,1,1,0,0,0,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [85]:
fingerprint = 'Graphonly'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
Graphonly_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
Graphonly_test.drop(['Name'],axis=1,inplace=True)
Graphonly_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/Graphonly_test_RRCK.csv', index=False)
Graphonly_test

,ID,SMILES,Permeability,GraphFP1,GraphFP2,GraphFP3,GraphFP4,GraphFP5,GraphFP6,GraphFP7,...,GraphFP1015,GraphFP1016,GraphFP1017,GraphFP1018,GraphFP1019,GraphFP1020,GraphFP1021,GraphFP1022,GraphFP1023,GraphFP1024
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [86]:
#KlekotaRothCount
fingerprint = 'KlekotaRothCount'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
KlekotaRothCount_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
KlekotaRothCount_train.drop(['Name'],axis=1,inplace=True)
KlekotaRothCount_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/KlekotaRothCount_train_RRCK.csv', index=False)
KlekotaRothCount_train

,ID,SMILES,Permeability,KRFPC1,KRFPC2,KRFPC3,KRFPC4,KRFPC5,KRFPC6,KRFPC7,...,KRFPC4851,KRFPC4852,KRFPC4853,KRFPC4854,KRFPC4855,KRFPC4856,KRFPC4857,KRFPC4858,KRFPC4859,KRFPC4860
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,19.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,18.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,18.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,18.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,18.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,8.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,8.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [87]:
fingerprint = 'KlekotaRothCount'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
KlekotaRothCount_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
KlekotaRothCount_test.drop(['Name'],axis=1,inplace=True)
KlekotaRothCount_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/KlekotaRothCount_test_RRCK.csv', index=False)
KlekotaRothCount_test

,ID,SMILES,Permeability,KRFPC1,KRFPC2,KRFPC3,KRFPC4,KRFPC5,KRFPC6,KRFPC7,...,KRFPC4851,KRFPC4852,KRFPC4853,KRFPC4854,KRFPC4855,KRFPC4856,KRFPC4857,KRFPC4858,KRFPC4859,KRFPC4860
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,19.0,5.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,18.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,10.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,10.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,9.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,12.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,8.0,0.0,0.0,0.0,0.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,7.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,5.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [88]:
#KlekotaRoth
fingerprint = 'KlekotaRoth'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_train.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
KlekotaRoth_train = pd.concat([df_train[['ID','SMILES','Permeability']], descriptors], axis=1)
KlekotaRoth_train.drop(['Name'],axis=1,inplace=True)
KlekotaRoth_train.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/KlekotaRoth_train_RRCK.csv', index=False)
KlekotaRoth_train

,ID,SMILES,Permeability,KRFP1,KRFP2,KRFP3,KRFP4,KRFP5,KRFP6,KRFP7,...,KRFP4851,KRFP4852,KRFP4853,KRFP4854,KRFP4855,KRFP4856,KRFP4857,KRFP4858,KRFP4859,KRFP4860
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [89]:
fingerprint = 'KlekotaRoth'
fingerprint_output_file = ''.join(['/home/users/akshay/PCPpred/RRCK/features/Fingerprints/',fingerprint,'_test.csv']) 
fingerprint_descriptortypes = fp[fingerprint]

padeldescriptor(mol_dir='/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi', 
                d_file=fingerprint_output_file, #'Substructure.csv'
                #descriptortypes='SubstructureFingerprint.xml', 
                descriptortypes= fingerprint_descriptortypes,
                detectaromaticity=True,
                standardizenitro=True,
                standardizetautomers=True,
                threads=2,
                removesalt=True,
                log=True,
                fingerprints=True)

descriptors = pd.read_csv(fingerprint_output_file)
KlekotaRoth_test = pd.concat([df_test[['ID','SMILES','Permeability']], descriptors], axis=1)
KlekotaRoth_test.drop(['Name'],axis=1,inplace=True)
KlekotaRoth_test.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/KlekotaRoth_test_RRCK.csv', index=False)
KlekotaRoth_test

,ID,SMILES,Permeability,KRFP1,KRFP2,KRFP3,KRFP4,KRFP5,KRFP6,KRFP7,...,KRFP4851,KRFP4852,KRFP4853,KRFP4854,KRFP4855,KRFP4856,KRFP4857,KRFP4858,KRFP4859,KRFP4860
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,1,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,1,0,0,0,0,0,0,...,0,0,0,0,0,1,0,1,0,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [90]:
df_train_morgan = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/morgan_fp_train_RRCK.csv')
df_train_morganCount = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/count_morgan_fp_train_RRCK.csv')
df_train_AP2d = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/AtomPairs2D_train_RRCK.csv')
df_train_AP2dCount = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/AtomPairs2DCount_train_RRCK.csv')
df_train_EState = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/EState_train_RRCK.csv')
df_train_Extended = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/Extended_train_RRCK.csv')
df_train_fp= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/Fingerprinter_train_RRCK.csv')
df_train_graph= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/Graphonly_train_RRCK.csv')
df_train_kr= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/KlekotaRoth_train_RRCK.csv')
df_train_krcount= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/KlekotaRothCount_train_RRCK.csv')
df_train_maccs= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/MACCS_train_RRCK.csv')
df_train_pubchem= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/PubChem_train_RRCK.csv')
df_train_Substr= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/Substructure_train_RRCK.csv')
df_train_Substrcount= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/SubstructureCount_train_RRCK.csv')

df = df_train_morgan.merge(df_train_morganCount, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_AP2d, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_AP2dCount, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_EState, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_Extended, on=['ID', 'SMILES', 'Permeability'], how='inner')
df = df.merge(df_train_fp, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_graph, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_kr, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_krcount, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_maccs, on=['ID', 'SMILES', 'Permeability'], how='inner')
df = df.merge(df_train_pubchem, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_Substr, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_train_Substrcount, on=['ID', 'SMILES', 'Permeability'], how='inner')
df

,ID,SMILES,Permeability,Morgan_fp_0,Morgan_fp_1,Morgan_fp_2,Morgan_fp_3,Morgan_fp_4,Morgan_fp_5,Morgan_fp_6,...,SubFPC298,SubFPC299,SubFPC300,SubFPC301,SubFPC302,SubFPC303,SubFPC304,SubFPC305,SubFPC306,SubFPC307
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,0,1,0,0,0,1,0,...,0.0,0.0,67.0,67.0,15.0,0.0,0.0,0.0,0.0,43.0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,0,1,0,0,0,1,0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,43.0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,0,1,0,0,0,1,0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,42.0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,0,1,0,0,0,1,0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,42.0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,0,1,0,0,0,1,0,...,0.0,0.0,65.0,65.0,14.0,0.0,0.0,0.0,0.0,42.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,0,1,0,0,0,1,0,...,0.0,0.0,34.0,34.0,7.0,0.0,0.0,0.0,0.0,26.0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,0,1,0,0,0,1,0,...,0.0,0.0,33.0,33.0,6.0,0.0,0.0,0.0,0.0,26.0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,0,1,0,0,0,1,0,...,0.0,0.0,34.0,34.0,6.0,0.0,0.0,0.0,0.0,20.0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,0,1,0,0,0,1,0,...,0.0,0.0,33.0,33.0,5.0,0.0,0.0,0.0,0.0,20.0


In [91]:
df.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Train/All_fingerprints_train_RRCK.csv',index=False)

In [92]:
df_test_morgan = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/morgan_fp_test_RRCK.csv')
df_test_morganCount = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/count_morgan_fp_test_RRCK.csv')
df_test_AP2d = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/AtomPairs2D_test_RRCK.csv')
df_test_AP2dCount = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/AtomPairs2DCount_test_RRCK.csv')
df_test_EState = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/EState_test_RRCK.csv')
df_test_Extended = pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/Extended_test_RRCK.csv')
df_test_fp= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/Fingerprinter_test_RRCK.csv')
df_test_graph= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/Graphonly_test_RRCK.csv')
df_test_kr= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/KlekotaRoth_test_RRCK.csv')
df_test_krcount= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/KlekotaRothCount_test_RRCK.csv')
df_test_maccs= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/MACCS_test_RRCK.csv')
df_test_pubchem= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/PubChem_test_RRCK.csv')
df_test_Substr= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/Substructure_test_RRCK.csv')
df_test_Substrcount= pd.read_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/SubstructureCount_test_RRCK.csv')

df = df_test_morgan.merge(df_test_morganCount, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_AP2d, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_AP2dCount, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_EState, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_Extended, on=['ID', 'SMILES', 'Permeability'], how='inner')
df = df.merge(df_test_fp, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_graph, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_kr, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_krcount, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_maccs, on=['ID', 'SMILES', 'Permeability'], how='inner')
df = df.merge(df_test_pubchem, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_Substr, on=['ID', 'SMILES', 'Permeability'], how='inner').merge(df_test_Substrcount, on=['ID', 'SMILES', 'Permeability'], how='inner')
df

,ID,SMILES,Permeability,Morgan_fp_0,Morgan_fp_1,Morgan_fp_2,Morgan_fp_3,Morgan_fp_4,Morgan_fp_5,Morgan_fp_6,...,SubFPC298,SubFPC299,SubFPC300,SubFPC301,SubFPC302,SubFPC303,SubFPC304,SubFPC305,SubFPC306,SubFPC307
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,0,1,0,0,0,1,0,...,0.0,0.0,67.0,67.0,15.0,0.0,0.0,0.0,0.0,43.0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,0,1,0,0,0,1,0,...,0.0,0.0,66.0,66.0,15.0,0.0,0.0,0.0,0.0,42.0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,0,0,0,0,0,1,0,...,0.0,0.0,54.0,54.0,12.0,0.0,0.0,0.0,0.0,36.0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,0,0,0,0,0,1,0,...,0.0,0.0,50.0,50.0,8.0,0.0,0.0,0.0,0.0,36.0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,0,0,0,0,0,1,0,...,0.0,0.0,49.0,49.0,11.0,0.0,0.0,0.0,0.0,33.0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,0,1,0,0,0,1,0,...,0.0,0.0,48.0,48.0,10.0,0.0,0.0,0.0,0.0,34.0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,0,0,0,0,0,1,0,...,0.0,0.0,47.0,47.0,13.0,0.0,0.0,0.0,0.0,30.0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,0,0,0,0,0,1,0,...,0.0,0.0,37.0,37.0,14.0,0.0,0.0,0.0,0.0,39.0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,0,0,0,0,0,1,0,...,0.0,0.0,48.0,48.0,16.0,0.0,0.0,0.0,0.0,29.0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,0,0,0,0,0,1,0,...,0.0,0.0,37.0,37.0,14.0,0.0,0.0,0.0,0.0,39.0


In [93]:
df.to_csv('/home/users/akshay/PCPpred/RRCK/features/Fingerprints/Test/All_fingerprints_test_RRCK.csv',index=False)

In [94]:
#Descriptors
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Descriptors, Descriptors3D
from rdkit.ML.Descriptors import MoleculeDescriptors
from tqdm import tqdm

In [95]:
calc = MoleculeDescriptors.MolecularDescriptorCalculator([desc[0] for desc in Chem.Descriptors._descList])

def calculate_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None 
    return calc.CalcDescriptors(mol)

In [96]:
#2D RDKit train dataset descriptors
df_train = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv")
descriptor_data = []
for smiles in df_train['SMILES']:
    descriptors = calculate_descriptors(smiles)
    if descriptors is not None:
        descriptor_data.append(descriptors)
    else:
        descriptor_data.append([np.nan] * len(calc.GetDescriptorNames()))

descriptor_df = pd.DataFrame(descriptor_data, columns=calc.GetDescriptorNames())
train_2d_rdkit = pd.concat([df_train[['ID','SMILES','Permeability']], descriptor_df], axis=1)
train_2d_rdkit.to_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Train_2d_RDKit_des_RRCK.csv',index=False)
print("Shape: ",train_2d_rdkit.shape, '\n')
train_2d_rdkit

Shape:  (140, 220) 



,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,15.193873,15.193873,0.130769,-1.621791,0.147476,26.802326,1216.662,...,0,0,0,0,0,0,0,0,0,0
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,15.152762,15.152762,0.128114,-1.816236,0.134993,26.372093,1214.646,...,0,0,0,0,0,0,0,0,0,0
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,15.129540,15.129540,0.022871,-1.609940,0.147925,26.905882,1202.635,...,0,0,0,0,0,0,0,0,0,0
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,15.028142,15.028142,0.097424,-1.231351,0.116062,27.411765,1202.635,...,0,0,0,0,0,0,0,0,0,0
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,15.068092,15.068092,0.128760,-1.611421,0.157205,27.083333,1188.608,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,13.851851,13.851851,0.023647,-1.031406,0.304960,27.600000,626.799,...,0,0,0,0,0,0,0,0,0,0
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,13.805528,13.805528,0.023859,-1.030988,0.318688,27.954545,612.772,...,0,0,0,0,0,0,0,0,0,0
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,13.850263,13.850263,0.054388,-0.925132,0.468423,29.209302,606.809,...,0,0,0,0,0,0,0,0,0,0
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,13.811409,13.811409,0.054806,-0.925415,0.488135,29.619048,592.782,...,0,0,0,0,0,0,0,0,0,0


In [97]:
#2D RDKit test dataset descriptors
df_test = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv")
descriptor_data = []
for smiles in df_test['SMILES']:
    descriptors = calculate_descriptors(smiles)
    if descriptors is not None:
        descriptor_data.append(descriptors)
    else:
        descriptor_data.append([np.nan] * len(calc.GetDescriptorNames()))

descriptor_df = pd.DataFrame(descriptor_data, columns=calc.GetDescriptorNames())
test_2d_rdkit = pd.concat([df_test[['ID','SMILES','Permeability']], descriptor_df], axis=1)
test_2d_rdkit.to_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Test_2d_RDKit_des_RRCK.csv',index=False)
print("Shape: ",test_2d_rdkit.shape, '\n')
test_2d_rdkit

Shape:  (35, 220) 



,ID,SMILES,Permeability,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,MolWt,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,15.144490,15.144490,0.113131,-1.744209,0.128505,27.116279,1218.634,...,0,0,0,0,0,0,0,0,0,0
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,15.129540,15.129540,0.022871,-1.609940,0.147925,26.905882,1202.635,...,0,0,0,0,0,0,0,0,0,0
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,14.943135,14.943135,0.009186,-1.218040,0.319427,27.500000,1095.438,...,0,0,0,0,0,0,0,0,1,0
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,14.843366,14.843366,0.011626,-1.216334,0.396119,28.337838,1039.330,...,0,0,0,0,0,0,0,0,0,0
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,14.986472,14.986472,0.005159,-1.183490,0.343286,27.394366,996.305,...,0,0,0,0,0,0,0,0,2,0
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,15.005167,15.005167,0.011136,-1.161318,0.363344,26.720588,953.280,...,0,0,0,0,0,0,0,0,0,0
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,14.929116,14.929116,0.022583,-1.148701,0.304153,26.582090,939.253,...,0,0,0,0,0,0,0,0,0,0
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,14.694489,14.694489,0.002536,-1.355433,0.153787,21.769231,914.089,...,0,0,0,0,0,1,0,0,0,0
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,14.686202,14.686202,0.009566,-1.067138,0.175293,25.412698,899.213,...,1,0,0,0,0,0,0,0,4,0
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,14.705070,14.705070,0.072633,-1.128302,0.149796,22.047619,876.137,...,0,0,0,0,0,1,0,0,0,0


In [98]:
from mordred import Calculator, descriptors
calc = Calculator(descriptors, ignore_3D=True)

def calculate_mordred_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None  # Return None if the SMILES is invalid
    return calc(mol)

In [99]:
print('start')
#2D Mordred train descriptors
df_train = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv")
descriptor_data = []
for smiles in df_train['SMILES']:
    descriptors = calculate_mordred_descriptors(smiles)
    if descriptors is not None:
        descriptor_data.append(descriptors)
    else:
        descriptor_data.append([np.nan] * len(calc.descriptors))

descriptor_df = pd.DataFrame(descriptor_data, columns=[desc.__class__.__name__ for desc in calc.descriptors])
train_mordred_2d = pd.concat([df_train[['ID','SMILES','Permeability']], descriptor_df], axis=1)
train_mordred_2d.to_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Train_2d_Mordred_desc_RRCK.csv', index=False)
print('Shape: ',train_mordred_2d.shape)
train_mordred_2d

start


/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Shape:  (140, 1616)


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix,AdjacencyMatrix,...,WalkCount,WalkCount,Weight,Weight,WienerIndex,WienerIndex,ZagrebIndex,ZagrebIndex,ZagrebIndex,ZagrebIndex
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1215.857018,6.109834,38268,152,418.0,484.0,44.111111,19.277778
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1213.841368,6.161631,38268,152,418.0,484.0,44.111111,19.277778
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.134733,2.444935,4.889759,...,11.208585,125.402718,1201.841368,6.131844,37337,150,412.0,477.0,43.250000,19.166667
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,1,100.268636,2.427280,4.854560,...,11.184019,125.341576,1201.841368,6.131844,37826,148,410.0,473.0,42.638889,19.277778
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,98.550044,2.444219,4.888335,...,11.198475,124.353468,1187.825718,6.154537,36408,148,408.0,472.0,43.000000,18.833333
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,55.201917,2.419901,4.786578,...,10.503834,95.786335,626.379183,6.593465,6693,72,224.0,257.0,18.027778,10.055556
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,53.640751,2.419606,4.784653,...,10.483550,94.692583,612.363533,6.656125,6341,70,220.0,252.0,17.777778,9.722222
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,52.045329,2.439503,4.844153,...,10.570008,93.767368,606.410483,6.251654,5725,76,214.0,251.0,20.250000,9.638889
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,50.591075,2.439146,4.842060,...,10.560671,92.694207,592.394833,6.302073,5381,75,210.0,247.0,20.000000,9.388889


In [100]:
#2D Mordred test descriptors
df_test = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv")
descriptor_data = []
for smiles in df_test['SMILES']:
    descriptors = calculate_mordred_descriptors(smiles)
    if descriptors is not None:
        descriptor_data.append(descriptors)
    else:
        descriptor_data.append([np.nan] * len(calc.descriptors))

descriptor_df = pd.DataFrame(descriptor_data, columns=[desc.__class__.__name__ for desc in calc.descriptors])
test_mordred_2d = pd.concat([df_test[['ID','SMILES','Permeability']], descriptor_df], axis=1)
test_mordred_2d.to_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Test_2d_Mordred_desc_RRCK.csv', index=False)
print('Shape: ',test_mordred_2d.shape)
test_mordred_2d

/home/users/akshay/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/core/fromnumeric.py:88: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


Shape:  (35, 1616)


,ID,SMILES,Permeability,ABCIndex,ABCGGIndex,AcidicGroupCount,BasicGroupCount,AdjacencyMatrix,AdjacencyMatrix,AdjacencyMatrix,...,WalkCount,WalkCount,Weight,Weight,WienerIndex,WienerIndex,ZagrebIndex,ZagrebIndex,ZagrebIndex,ZagrebIndex
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.888651,2.446436,4.892748,...,11.228478,126.484444,1217.836283,6.181910,38268,152,418.0,484.0,44.111111,19.277778
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,100.134733,2.444935,4.889759,...,11.208585,125.402718,1201.841368,6.131844,37337,150,412.0,477.0,43.250000,19.166667
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,96.282174,2.460904,4.898232,...,11.232960,131.870970,1094.710354,6.364595,28969,148,388.0,464.0,36.055556,18.000000
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,91.172927,2.459349,4.893348,...,11.208477,127.699425,1038.647754,6.491548,25441,143,372.0,447.0,35.055556,16.916667
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,87.929966,2.459011,4.889438,...,11.129422,124.389878,995.641940,6.382320,22326,133,354.0,422.0,32.472222,16.250000
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,82.784106,2.466654,4.910608,...,11.091041,121.222371,952.636126,6.267343,19257,124,344.0,406.0,32.333333,15.000000
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,83.484783,2.465318,4.908864,...,11.066935,120.067649,938.620476,6.299466,18483,125,332.0,396.0,29.638889,15.583333
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,82.177160,2.435750,4.819009,...,10.908540,119.586829,913.400825,7.486892,18371,100,334.0,383.0,21.673611,14.208333
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,78.841373,2.449488,4.853049,...,10.792633,115.248549,898.535032,6.558650,16763,102,302.0,347.0,24.361111,14.777778
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,module 'numpy' has no attribute 'float'.\n`np....,module 'numpy' has no attribute 'float'.\n`np....,0,0,80.291949,2.435508,4.818260,...,10.844022,117.353041,875.440404,7.060003,16811,95,322.0,368.0,19.951389,13.847222


In [101]:
#RDKit 3d descriptors
def generate_3d_descriptors(smiles):
    # Convert SMILES to a molecule object
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Invalid SMILES: {smiles}")
        return None
    
    # Add hydrogens
    mol = Chem.AddHs(mol)
    
    # Generate 3D coordinates for the molecule
    AllChem.EmbedMolecule(mol)
    
    try:
        descriptors = Descriptors3D.CalcMolDescriptors3D(mol)
        return descriptors
    except Exception as e:
        print(f"Error calculating descriptors for SMILES '{smiles}': {e}")
        return None

In [102]:
df_train = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.csv")
descriptor_data = []
for smiles in tqdm(df_train['SMILES'],desc='3d_descriptors', unit='smiles'):
    descriptors =generate_3d_descriptors(smiles) 
    if descriptors is not None:
        descriptor_data.append(descriptors)
    else:
        descriptor_data.append({'PMI1': np.nan,
  'PMI2': np.nan,
  'PMI3': np.nan,
  'NPR1': np.nan,
  'NPR2': np.nan,
  'RadiusOfGyration': np.nan,
  'InertialShapeFactor': np.nan,
  'Eccentricity': np.nan,
  'Asphericity': np.nan,
  'SpherocityIndex': np.nan,
  'PBF': np.nan})

3d_descriptors: 100%|██████████| 140/140 [01:20<00:00,  1.73smiles/s]


In [103]:
descriptor_df = pd.DataFrame(descriptor_data)
descriptor_df

,PMI1,PMI2,PMI3,NPR1,NPR2,RadiusOfGyration,InertialShapeFactor,Eccentricity,Asphericity,SpherocityIndex,PBF
0,25557.550768,38719.861789,59078.304374,0.432605,0.655399,7.119995,0.000026,0.901584,0.224932,0.159022,1.434147
1,25195.527353,38098.222481,54940.535676,0.458596,0.693445,6.976408,0.000028,0.888645,0.190983,0.276159,1.867453
2,24627.941709,37770.489101,52166.145232,0.472106,0.724042,6.901496,0.000029,0.881542,0.173457,0.300184,1.966338
3,27053.583076,35398.003591,54685.502768,0.494712,0.647301,6.978552,0.000024,0.869057,0.175665,0.268855,1.757097
4,25621.505581,37274.558430,58032.092029,0.441506,0.642309,7.132297,0.000025,0.897258,0.221165,0.149807,1.434163
...,...,...,...,...,...,...,...,...,...,...,...
135,7624.940741,7922.483693,14006.527516,0.544385,0.565628,4.855440,0.000074,0.838836,0.178213,0.186149,1.117525
136,6232.953683,8749.353631,12635.210560,0.493300,0.692458,4.747094,0.000111,0.869859,0.163679,0.271461,1.309583
137,5787.868928,7241.500975,11269.076352,0.513606,0.642599,4.474538,0.000111,0.858026,0.163878,0.237706,1.199945
138,5518.344689,6698.311541,10323.461033,0.534544,0.648844,4.360292,0.000118,0.845141,0.148106,0.299445,1.277044


In [104]:
num_columns = descriptor_df.shape[1]
descriptor_df.columns = [f'3d_rdkit_{i+1}' for i in range(num_columns)]
train_3d_rdkit = pd.concat([df_train[['ID','SMILES','Permeability']], descriptor_df], axis=1)
print('Shape before:' , train_3d_rdkit.shape)
train_3d_rdkit.to_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Train_3d_RDKit_desc_RRCK.csv',index=False)
print('Shape after: ',train_3d_rdkit.shape)
train_3d_rdkit

Shape before: (140, 14)
Shape after:  (140, 14)


,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,3d_rdkit_8,3d_rdkit_9,3d_rdkit_10,3d_rdkit_11
0,2358,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C(...,-6.13,25557.550768,38719.861789,59078.304374,0.432605,0.655399,7.119995,0.000026,0.901584,0.224932,0.159022,1.434147
1,2359,C/C=C/C[C@@H](C)C(=O)[C@H]1C(=O)N[C@@H](C(C)C)...,-6.66,25195.527353,38098.222481,54940.535676,0.458596,0.693445,6.976408,0.000028,0.888645,0.190983,0.276159,1.867453
2,5669,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.76,24627.941709,37770.489101,52166.145232,0.472106,0.724042,6.901496,0.000029,0.881542,0.173457,0.300184,1.966338
3,2360,C/C=C/C[C@@H](C)[C@H]1OC(=O)[C@H](C(C)C)N(C)C(...,-6.78,27053.583076,35398.003591,54685.502768,0.494712,0.647301,6.978552,0.000024,0.869057,0.175665,0.268855,1.757097
4,2353,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](C)...,-5.87,25621.505581,37274.558430,58032.092029,0.441506,0.642309,7.132297,0.000025,0.897258,0.221165,0.149807,1.434163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
135,2335,CC[C@H]1NC(=O)[C@H](CC(C)C)NC(=O)[C@H](Cc2cccc...,-5.53,7624.940741,7922.483693,14006.527516,0.544385,0.565628,4.855440,0.000074,0.838836,0.178213,0.186149,1.117525
136,2334,CC(C)C[C@@H]1NC(=O)[C@H](Cc2ccccc2)NC(=O)[C@H]...,-5.58,6232.953683,8749.353631,12635.210560,0.493300,0.692458,4.747094,0.000111,0.869859,0.163679,0.271461,1.309583
137,2305,CCC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)...,-4.85,5787.868928,7241.500975,11269.076352,0.513606,0.642599,4.474538,0.000111,0.858026,0.163878,0.237706,1.199945
138,2304,CC[C@@H]1C(=O)N(C)[C@@H](C)C(=O)N[C@@H](CC(C)C...,-5.51,5518.344689,6698.311541,10323.461033,0.534544,0.648844,4.360292,0.000118,0.845141,0.148106,0.299445,1.277044


In [105]:
df_test = pd.read_csv("/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.csv")
descriptor_data = []
for smiles in tqdm(df_test['SMILES'],desc='3d_descriptors', unit='smiles'):
    descriptors = generate_3d_descriptors(smiles)
    if descriptors is not None:
        descriptor_data.append(descriptors)
    else:
        descriptor_data.append({'PMI1': np.nan,
  'PMI2': np.nan,
  'PMI3': np.nan,
  'NPR1': np.nan,
  'NPR2': np.nan,
  'RadiusOfGyration': np.nan,
  'InertialShapeFactor': np.nan,
  'Eccentricity': np.nan,
  'Asphericity': np.nan,
  'SpherocityIndex': np.nan,
  'PBF': np.nan})

descriptor_data

3d_descriptors: 100%|██████████| 35/35 [00:19<00:00,  1.81smiles/s]


[{'PMI1': 25465.77229741613,
  'PMI2': 41635.23635727709,
  'PMI3': 57764.58127438965,
  'NPR1': 0.4408544429059432,
  'NPR2': 0.7207744856576391,
  'RadiusOfGyration': 7.157638221080254,
  'InertialShapeFactor': 2.830365705149936e-05,
  'Eccentricity': 0.8975786094655388,
  'Asphericity': 0.20072845751791388,
  'SpherocityIndex': 0.2746179634758924,
  'PBF': 1.8948536483847755},
 {'PMI1': 23425.027477352316,
  'PMI2': 36819.01451869982,
  'PMI3': 51958.19174897247,
  'NPR1': 0.45084377821550287,
  'NPR2': 0.7086277116144627,
  'RadiusOfGyration': 6.829970600182565,
  'InertialShapeFactor': 3.025088070012191e-05,
  'Eccentricity': 0.8926028723034508,
  'Asphericity': 0.1942492485392814,
  'SpherocityIndex': 0.2791227585288796,
  'PBF': 1.8494523457490875},
 {'PMI1': 19228.307860212684,
  'PMI2': 26584.788949910257,
  'PMI3': 41557.971167824624,
  'NPR1': 0.4626863949292066,
  'NPR2': 0.6397037247692439,
  'RadiusOfGyration': 6.315022804780913,
  'InertialShapeFactor': 3.326885181056010

In [106]:
descriptor_df = pd.DataFrame(descriptor_data)

In [107]:
num_columns = descriptor_df.shape[1]
print('num_columns',num_columns)
descriptor_df.columns = [f'3d_rdkit_{i+1}' for i in range(num_columns)]
print('descriptor_df.columns',descriptor_df.columns)
test_3d_rdkit = pd.concat([df_test[['ID','SMILES','Permeability']], descriptor_df], axis=1)
print('Shape before:' , test_3d_rdkit.shape)
test_3d_rdkit.to_csv('/home/users/akshay/PCPpred/RRCK/features/Descriptors/Test_3d_RDKit_desc_RRCK.csv',index=False)
print('Shape after: ',test_3d_rdkit.shape)
test_3d_rdkit

num_columns 11
descriptor_df.columns Index(['3d_rdkit_1', '3d_rdkit_2', '3d_rdkit_3', '3d_rdkit_4', '3d_rdkit_5',
       '3d_rdkit_6', '3d_rdkit_7', '3d_rdkit_8', '3d_rdkit_9', '3d_rdkit_10',
       '3d_rdkit_11'],
      dtype='object')
Shape before: (35, 14)
Shape after:  (35, 14)


,ID,SMILES,Permeability,3d_rdkit_1,3d_rdkit_2,3d_rdkit_3,3d_rdkit_4,3d_rdkit_5,3d_rdkit_6,3d_rdkit_7,3d_rdkit_8,3d_rdkit_9,3d_rdkit_10,3d_rdkit_11
0,2352,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H]([C...,-6.340,25465.772297,41635.236357,57764.581274,0.440854,0.720774,7.157638,0.000028,0.897579,0.200728,0.274618,1.894854
1,2357,C/C=C/C[C@@H](C)[C@@H](O)[C@H]1C(=O)N[C@@H](CC...,-5.950,23425.027477,36819.014519,51958.191749,0.450844,0.708628,6.829971,0.000030,0.892603,0.194249,0.279123,1.849452
2,1883,CCCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@H](C)C...,-6.240,19228.307860,26584.788950,41557.971168,0.462686,0.639704,6.315023,0.000033,0.886522,0.203552,0.190402,1.402717
3,1882,CCC[C@H]1C(=O)N(C)[C@H](CC)C(=O)N(C)[C@@H](C)C...,-7.000,18709.773342,27134.585978,41923.420928,0.446285,0.647242,6.497942,0.000035,0.894891,0.215122,0.176479,1.403879
4,1879,CCCC[C@H]1C(=O)N(C)[C@H](C)C(=O)N2CCC[C@@H]2C(...,-6.210,15494.460333,25480.222143,35133.289000,0.441019,0.725244,6.180220,0.000047,0.897498,0.199772,0.263703,1.586357
5,1868,CC(C)C[C@H]1C(=O)N(C)[C@H](CC(C)C)C(=O)N2CCC[C...,-4.990,12652.784435,21781.115461,29110.712319,0.434644,0.748217,5.773167,0.000059,0.900603,0.202041,0.300452,1.566055
6,1876,CCC[C@H]1C(=O)N(C)[C@H](CCC)C(=O)N(C)[C@H](CCC...,-5.140,14906.294395,19726.009637,30917.986108,0.482124,0.638011,5.907191,0.000043,0.876103,0.188447,0.228130,1.409108
7,1859,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.210,14125.144457,25108.706249,32442.299466,0.435393,0.773950,6.261496,0.000055,0.900241,0.198517,0.306028,1.607786
8,5662,CCCC[C@@H]1NC(=O)[C@H](CCCC)NC(=O)[C@H](CCCC)N...,-6.100,12577.621436,20415.610939,26519.217510,0.474283,0.769842,5.752513,0.000061,0.880372,0.165488,0.338583,1.732709
9,1853,CC(C)(C)C[C@@H]1NC(=O)[C@@H](Cc2ccccc2)NC(=O)[...,-5.400,11732.937028,20116.819627,25872.348901,0.453493,0.777541,5.739447,0.000066,0.891260,0.182085,0.382889,1.762876


In [112]:
import os

#Train MOL structure generator
input_file = '/home/users/akshay/PCPpred/RRCK/data/Train_RRCK.smi'
output_dir = '/home/users/akshay/PCPpred/RRCK/data/Train_mol_RRCK'

os.makedirs(output_dir, exist_ok=True)

# Read the .smi file
with open(input_file, 'r') as f:
    lines = f.readlines()

for line in tqdm(lines, desc="Processing molecules"):
    parts = line.strip().split()
    if len(parts) != 2:
        continue  
    smiles, ID = parts

    # Generate 3D conformation
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Invalid SMILES: {smiles} for id: {ID}")
        continue

    # Add hydrogens and generate 3D coordinates
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    

    # Save the 3D conformation in MDL format
    output_file = os.path.join(output_dir, f"{ID}.mdl")
    with open(output_file, 'w') as out:
        out.write(Chem.MolToMolBlock(mol))

print("3D conformations generated and saved.")

Processing molecules: 100%|██████████| 140/140 [01:16<00:00,  1.82it/s]

3D conformations generated and saved.


In [113]:
directory_path = '/home/users/akshay/PCPpred/RRCK/data/Train_mol_RRCK'
files = os.listdir(directory_path)
file_count = sum(os.path.isfile(os.path.join(directory_path, f)) for f in files)

print(f"Number of files in the directory: {file_count}")

Number of files in the directory: 140


In [114]:
#Test MOL structure generator
input_file = '/home/users/akshay/PCPpred/RRCK/data/Test_RRCK.smi'
output_dir = '/home/users/akshay/PCPpred/RRCK/data/Test_mol_RRCK'

os.makedirs(output_dir, exist_ok=True)

# Read the .smi file
with open(input_file, 'r') as f:
    lines = f.readlines()

for line in tqdm(lines, desc="Processing molecules"):
    parts = line.strip().split()
    if len(parts) != 2:
        continue  
    smiles, ID = parts

    # Generate 3D conformation
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Invalid SMILES: {smiles} for id: {ID}")
        continue

    # Add hydrogens and generate 3D coordinates
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    

    # Save the 3D conformation in MDL format
    output_file = os.path.join(output_dir, f"{ID}.mdl")
    with open(output_file, 'w') as out:
        out.write(Chem.MolToMolBlock(mol))

print("3D conformations generated and saved.")

Processing molecules: 100%|██████████| 35/35 [00:21<00:00,  1.64it/s]

3D conformations generated and saved.


In [115]:
directory_path = '/home/users/akshay/PCPpred/RRCK/data/Test_mol_RRCK'
files = os.listdir(directory_path)
file_count = sum(os.path.isfile(os.path.join(directory_path, f)) for f in files)

print(f"Number of files in the directory: {file_count}")

Number of files in the directory: 35
